# Clasificación Naive Bayes usando el conjunto de datos de Vino(UCI ML Repo)

La técnica de clasificación [Naive Bayes](http://dataaspirant.com/2017/02/06/naive-bayes-classifier-machine-learning/) se basa en el llamado teorema bayesiano y es particularmente adecuada cuando la dimensionalidad de las entradas son altas. A pesar de su simplicidad, Naive Bayes a menudo puede superar a los métodos de clasificación más sofisticados.

### Teorema de Bayes

El algoritmo se basa en el famoso [___Teorema de Bayes___](https://en.wikipedia.org/wiki/Bayes%27_theorem) que lleva el nombre del Rev. Thomas Bayes. Funciona con probabilidad condicional. [Probabilidad condicional](https://en.wikipedia.org/wiki/Conditional_probability) es la probabilidad de que suceda algo, dado que ya ha ocurrido algo más. Usando la probabilidad condicional, podemos calcular la probabilidad de un evento usando su conocimiento previo.

El teorema de Bayes se establece matemáticamente como la siguiente ecuación:

$${\displaystyle P(A\mid B)={\frac {P(B\mid A)\,P(A)}{P(B)}},}$$
donde $ A $ y $ B $ son eventos y $ P (B) \neq {0} $.

$P(A \mid B) $ es una [probabilidad condicional](https://en.wikipedia.org/wiki/Conditional_probability): la probabilidad de que ocurra el evento $ A $ dado que $ B $ es verdadero.

$P(B \mid A) $ también es una probabilidad condicional: la probabilidad de que ocurra el evento $ B $ dado que $ A $ es verdadero.

$P(A)$ y $P(B) $ son las probabilidades de observar $ A $ y $ B $ independientemente entre sí; esto se conoce como la [probabilidad marginal](https://en.wikipedia.org/wiki/Marginal_probability).

### ¿Qué es _Naive_ en Naive Bayes y por qué es un algoritmo súper rápido?

Se le llama Bayes ingenuo(Naive) o Bayes idiota(Idiot) porque el cálculo de las probabilidades de cada hipótesis se simplifica para hacer su cálculo manejable. En lugar de intentar calcular los valores de cada valor de atributo, se supone que son condicionalmente independientes dado el valor objetivo.

Esta es una suposición muy sólida que es muy poco probable en datos reales, es decir, que los atributos no interactúan. Sin embargo, el enfoque funciona sorprendentemente bien en datos donde este supuesto no se cumple.

El entrenamiento es rápido porque solo es necesario calcular la probabilidad de cada clase y la probabilidad de cada clase dados diferentes valores de entrada. **No es necesario ajustar coeficientes mediante procedimientos de optimización.**

Las probabilidades de clase son simplemente la frecuencia de instancias que pertenecen a cada clase dividida por el número total de instancias. Las probabilidades condicionales son la frecuencia de cada valor de atributo para un valor de clase determinado dividido por la frecuencia de instancias con ese valor de clase.

### Datos analizados en este cuaderno

En este cuaderno, mostraremos cómo utilizar el método Naive Bayes de Python scikit-learn para clasificar el origen del vino en base a datos de análisis físico-químicos. Estos datos son el resultado de un análisis químico de vinos cultivados en la misma región en Italia pero derivados de tres cultivares diferentes. El análisis determinó las cantidades de 13 componentes que se encuentran en cada uno de los tres tipos de vinos.

Los detalles se pueden [**encontrar aquí**](http://archive.ics.uci.edu/ml/datasets/Wine).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Leemos los datos y realizamos un análisis exploratorio básico

#### Conjunto de datos

In [ ]:
df = pd.read_csv('https://github.com/ulewis/Ejemplos/raw/main/Datos/wine.data.csv')
df.head(10)

#### Estadísticas básicas de las características

In [ ]:
df.iloc[:,1:].describe().T

#### Boxplots por salida de etiquetas/clases

In [ ]:
for c in df.columns[1:]:
    df.boxplot(c,by='Class',figsize=(7,4),fontsize=14)
    plt.title("{}\n".format(c),fontsize=16)
    plt.xlabel("Wine Class", fontsize=16)

**Se puede ver que algunas características clasifican las etiquetas de los vinos con bastante claridad.** Por ejemplo, la alcalinidad, los fenoles totales o los flavonoides producen diagramas de caja con medianas bien separadas, que son claramente indicativas de las clases de vinos.

A continuación se muestra un ejemplo de separación de clases usando dos variables

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['OD280/OD315 of diluted wines'],df['Flavanoids'],c=df['Class'],edgecolors='k',alpha=0.8,s=100)
plt.grid(True)
plt.title("Scatter plot of two features showing the \ncorrelation and class seperation",fontsize=15)
plt.xlabel("OD280/OD315 of diluted wines",fontsize=15)
plt.ylabel("Flavanoids",fontsize=15)

#### ¿Son las funciones independientes? Graficamos la  matriz de covarianza

Se puede ver que existe una buena cantidad de correlación entre las características, es decir, no son independientes entre sí, como se supone en la técnica Naive Bayes. Sin embargo, seguiremos adelante y aplicaremos el clasificador para ver su rendimiento.

In [ ]:
def correlation_matrix(df):
    from matplotlib import pyplot as plt
    from matplotlib import cm as cm

    fig = plt.figure(figsize=(16,12))
    ax1 = fig.add_subplot(111)
    cmap = cm.get_cmap('jet', 30)
    cax = ax1.imshow(df.corr(), interpolation="nearest", cmap=cmap)
    ax1.grid(True)
    plt.title('Wine data set features correlation\n',fontsize=15)
    labels=df.columns
    ax1.set_xticklabels(labels,fontsize=9)
    ax1.set_yticklabels(labels,fontsize=9)
    # Add colorbar, make sure to specify tick locations to match desired ticklabels
    fig.colorbar(cax, ticks=[0.1*i for i in range(-11,11)])
    plt.show()

correlation_matrix(df)

## Clasificación Naive Bayes

#### Dividimos el conjunto de entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split

test_size=0.3 # Test-set fraction

In [ ]:
X = df.drop('Class',axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size)

In [ ]:
X_train.shape

In [ ]:
X_train.head()

#### Clasificación usando GaussianNB

Dada una variable de clase $ y $ y un vector de característica dependiente $ x_1 $ a $ x_n $, el teorema de Bayes establece la siguiente relación:

$$P(y \mid x_1, \dots, x_n) = \frac{P(y) P(x_1, \dots x_n \mid y)} {P(x_1, \dots, x_n)}$$
Utilizando la suposición ingenua de independencia de que
$$P(x_i | y, x_1, \dots, x_{i-1}, x_{i+1}, \dots, x_n) = P(x_i | y),$$
para todo $ i $, esta relación se simplifica a
$$P(y \mid x_1, \dots, x_n) = \frac{P(y) \prod_{i=1}^{n} P(x_i \mid y)} {P(x_1, \dots, x_n)}$$

Dado que $ P (x_1, \dots, x_n) $ es constante dada la entrada, podemos usar la siguiente regla de clasificación:
$$P(y \mid x_1, \dots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)$$
$$\Downarrow$$
$$\hat{y} = \arg\max_y P(y) \prod_{i=1}^{n} P(x_i \mid y),$$

y podemos usar la estimación de [**Maximum A Posteriori**](https://en.wikipedia.org/wiki/Maximum_a_posteriori_estimation) (MAP) para estimar $ P (y) $ y $ P (x_i \mid y) $ ; la primera es entonces la frecuencia relativa de la clase $ y $ en el conjunto de entrenamiento.

***GaussianNB ()*** implementa el algoritmo Gaussian Naive Bayes para la clasificación. **Se supone que la probabilidad de las características es gaussiana**:

$$ P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma^2_y}} \exp(-\frac{(x_i - \mu_y)^2}{2\sigma^2_y}) $$

Los parámetros $ \sigma_y $ y $ \mu_y $ se estiman utilizando la máxima verosimilitud.

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
nbc = GaussianNB()

In [ ]:
nbc.fit(X_train,y_train)

#### Predicción, reporte de clasificación y matriz de confusión

In [ ]:
y_pred = nbc.predict(X_test)
mislabel = np.sum((y_test!=y_pred))
print("La cantidad total de puntos de datos mal etiquetados de {} muestras de prueba es {}".format(len(y_test),mislabel))

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print("El informe de clasificación es el siguiente...\n")
print(classification_report(y_pred,y_test))

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
cm = (confusion_matrix(y_test,y_pred))
cmdf = pd.DataFrame(cm,index=['Class 1','Class 2',' Class 3'], columns=['Class 1','Class 2',' Class 3'])
print("La matriz de confusión es la siguiente...\n")
cmdf

**Esto mostró que incluso en presencia de correlación entre las características, el algoritmo Naive Bayes funcionó bastante bien y podría separar las clases fácilmente**